# LSTM Prediction
LSTM, CNN, and CNN-LSTM deep learning models for daily temperature prediction.

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
from scipy.ndimage import uniform_filter1d
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb
import lightgbm as lgb
import tensorflow as tf
from tensorflow.keras import layers, Sequential
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
import warnings, glob
warnings.filterwarnings('ignore')

# Global constants
EXTREME_T   = 28        # Extreme heat threshold (°C)
MAN_LAT     = 53.48
MAN_LON_360 = 360 - 2.24  # Manchester longitude in 0-360 format


In [ ]:
# ─── Global plot settings (journal style) ───────────────────────────────────
matplotlib.rcParams.update({
    'font.family':      'DejaVu Sans',
    'font.size':        11,
    'axes.titlesize':   12,
    'axes.labelsize':   11,
    'xtick.labelsize':  10,
    'ytick.labelsize':  10,
    'legend.fontsize':  9,
    'figure.dpi':       130,
    'savefig.dpi':      300,
    'savefig.bbox':     'tight',
    'axes.spines.top':  False,
    'axes.spines.right':False,
    'axes.grid':        True,
    'grid.alpha':       0.22,
    'grid.linestyle':   '--',
    'axes.axisbelow':   True,
})

C = dict(
    RF='#2166AC', XGB='#4DAC26', LGB='#E08214',
    LSTM='#762A83', CNN='#D6604D', CNN_LSTM='#01665E',
    Ensemble='#543005', obs='#252525',
    hist='#4393C3', fut='#D6604D', extreme='#B2182B',
)

def savefig(name):
    plt.savefig(f'{name}.png', dpi=300, bbox_inches='tight', facecolor='white')


## Prerequisites
Run `1_data_processing.ipynb` first.

In [145]:
# =============================================================================
# PART 6  深度学习（FIX-4/5: 正确滑窗 + 时序数据 + 统一 scaler）
# =============================================================================
print('\n' + '=' * 70)
print('PART 6 — Deep Learning  [FIX-4: proper time-series sequences]')
print('=' * 70)
 
LOOKBACK   = 14   # 过去 14 天
BATCH_SIZE = 256
MAX_EPOCHS = 60
PATIENCE   = 12
 
def make_sequences(X_arr, y_arr, lookback):
    """
    滑动窗口：X[i:i+lookback] → y[i+lookback]
    输入必须保持时间顺序（不能 shuffle）
    """
    Xs, ys = [], []
    for i in range(len(X_arr) - lookback):
        Xs.append(X_arr[i:i+lookback])
        ys.append(y_arr[i+lookback])
    return np.array(Xs, dtype=np.float32), np.array(ys, dtype=np.float32)
 
# 注意：必须用已按时间排好序的数据
X_tr_seq, y_tr_seq   = make_sequences(X_train_s, y_train_s, LOOKBACK)
X_va_seq, y_va_seq   = make_sequences(X_val_s,   y_val_s,   LOOKBACK)
X_te_seq, y_te_seq   = make_sequences(X_test_s,  y_test_s,  LOOKBACK)
 
# 对应的原始°C目标（用于评估）
y_te_seq_C = y_test_C[LOOKBACK:]   # 序列化后样本数少了 LOOKBACK 个
 
print(f'  Seq shapes: train{X_tr_seq.shape} val{X_va_seq.shape} test{X_te_seq.shape}')
N_FEAT = X_tr_seq.shape[2]
 
cb_list = [
    EarlyStopping(monitor='val_loss', patience=PATIENCE,
                  restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                      patience=6, min_lr=1e-6, verbose=0),
]
 
# ── LSTM ──────────────────────────────────────────────────────────────────────
print('\n[DL 1/3] LSTM ...')
model_lstm = Sequential([
    layers.LSTM(64, activation='tanh', return_sequences=True,
                dropout=0.15, input_shape=(LOOKBACK, N_FEAT)),
    layers.Dropout(0.15),
    layers.LSTM(32, activation='tanh', dropout=0.15),
    layers.Dropout(0.15),
    layers.Dense(32, activation='relu'),
    layers.Dense(1),
], name='LSTM')
model_lstm.compile(optimizer=Adam(1e-3), loss='mse')
history_lstm = model_lstm.fit(
    X_tr_seq, y_tr_seq,
    validation_data=(X_va_seq, y_va_seq),
    epochs=MAX_EPOCHS, batch_size=BATCH_SIZE,
    callbacks=cb_list, verbose=0)
 
y_pred_lstm_C = inv_y(model_lstm.predict(X_te_seq, verbose=0))
lstm_metrics  = evaluate_C(y_te_seq_C, y_pred_lstm_C, 'LSTM')
lstm_ext      = extreme_scores(y_te_seq_C, y_pred_lstm_C)
print(f'  Trained {len(history_lstm.history["loss"])} epochs')
tf.keras.backend.clear_session()


PART 6 — Deep Learning  [FIX-4: proper time-series sequences]
  Seq shapes: train(19299, 14, 46) val(4001, 14, 46) test(4001, 14, 46)

[DL 1/3] LSTM ...
Epoch 18: early stopping
Restoring model weights from the end of the best epoch: 6.
  LSTM               MAE=1.879°C  RMSE=2.452°C  R²=0.8032  MAPE=11.82%
  Trained 18 epochs


In [146]:
# ── CNN ───────────────────────────────────────────────────────────────────────
print('\n[DL 2/3] CNN ...')
model_cnn = Sequential([
    layers.Conv1D(64, kernel_size=3, activation='relu',
                  padding='causal', input_shape=(LOOKBACK, N_FEAT)),
    layers.Conv1D(32, kernel_size=3, activation='relu', padding='causal'),
    layers.GlobalAveragePooling1D(),
    layers.Dense(32, activation='relu'),
    layers.Dropout(0.15),
    layers.Dense(1),
], name='CNN')
model_cnn.compile(optimizer=Adam(1e-3), loss='mse')
history_cnn = model_cnn.fit(
    X_tr_seq, y_tr_seq,
    validation_data=(X_va_seq, y_va_seq),
    epochs=MAX_EPOCHS, batch_size=BATCH_SIZE,
    callbacks=cb_list, verbose=0)
 
y_pred_cnn_C = inv_y(model_cnn.predict(X_te_seq, verbose=0))
cnn_metrics  = evaluate_C(y_te_seq_C, y_pred_cnn_C, 'CNN')
cnn_ext      = extreme_scores(y_te_seq_C, y_pred_cnn_C)
print(f'  Trained {len(history_cnn.history["loss"])} epochs')
tf.keras.backend.clear_session()
 


[DL 2/3] CNN ...
Epoch 52: early stopping
Restoring model weights from the end of the best epoch: 40.
  CNN                MAE=1.771°C  RMSE=2.243°C  R²=0.8353  MAPE=11.52%
  Trained 52 epochs


In [147]:
# ── CNN-LSTM ──────────────────────────────────────────────────────────────────
print('\n[DL 3/3] CNN-LSTM ...')
model_cnnlstm = Sequential([
    layers.Conv1D(64, kernel_size=3, activation='relu',
                  padding='causal', input_shape=(LOOKBACK, N_FEAT)),
    layers.MaxPooling1D(pool_size=2),
    layers.Dropout(0.15),
    layers.LSTM(32, activation='tanh', dropout=0.15),
    layers.Dense(32, activation='relu'),
    layers.Dropout(0.15),
    layers.Dense(1),
], name='CNN_LSTM')
model_cnnlstm.compile(optimizer=Adam(1e-3), loss='mse')
history_cnnlstm = model_cnnlstm.fit(
    X_tr_seq, y_tr_seq,
    validation_data=(X_va_seq, y_va_seq),
    epochs=MAX_EPOCHS, batch_size=BATCH_SIZE,
    callbacks=cb_list, verbose=0)
 
y_pred_cnnlstm_C = inv_y(model_cnnlstm.predict(X_te_seq, verbose=0))
cnnlstm_metrics  = evaluate_C(y_te_seq_C, y_pred_cnnlstm_C, 'CNN-LSTM')
cnnlstm_ext      = extreme_scores(y_te_seq_C, y_pred_cnnlstm_C)
print(f'  Trained {len(history_cnnlstm.history["loss"])} epochs')
 
dl_metrics_df = pd.DataFrame({
    'LSTM': lstm_metrics, 'CNN': cnn_metrics, 'CNN-LSTM': cnnlstm_metrics}).T
print('\nDL model summary (°C):')
print(dl_metrics_df.to_string())


[DL 3/3] CNN-LSTM ...
Epoch 55: early stopping
Restoring model weights from the end of the best epoch: 43.
  CNN-LSTM           MAE=1.672°C  RMSE=2.148°C  R²=0.8490  MAPE=10.66%
  Trained 55 epochs

DL model summary (°C):
               MAE      RMSE        R2       MAPE
LSTM      1.878999  2.451916  0.803228  11.824012
CNN       1.770706  2.242895  0.835347  11.520087
CNN-LSTM  1.672209  2.148207  0.848956  10.663186
